In [ ]:
from glasspy.data import SciGlass

source = SciGlass(
    properties_cfg={"keep_all": True},
    compounds_cfg={"keep_all": True},
)

df_full = source.data

# Check how many rows have each property
property_counts = df_full["property"].notna().sum().sort_values(ascending=False)
print(property_counts.head(20))

In [ ]:
# Check the compound columns (oxides)
compounds_cols = df_full["compounds"].columns.tolist()
print(f"Number of oxides: {len(compounds_cols)}")
print(compounds_cols)

# Check Tg
print(df_full["property"]["Tg"].describe())

In [ ]:
compounds = df_full["compounds"]

# What % of values are non-zero per column
non_zero_pct = (compounds > 0).mean().sort_values(ascending=False)
print(non_zero_pct.head(20))

In [ ]:
threshold = 0.05
common_compounds = non_zero_pct[non_zero_pct > threshold].index.tolist()
print(f"Compounds appearing in >5% of compositions: {len(common_compounds)}")
print(common_compounds)

In [ ]:
oxide_features = [
    'SiO2', 'B2O3', 'Al2O3', 'Na2O', 'CaO', 'K2O', 'MgO', 'P2O5',
    'BaO', 'Li2O', 'ZnO', 'TiO2', 'PbO', 'ZrO2', 'SrO', 'La2O3',
    'Fe2O3', 'Nb2O5'
]

compounds = df_full["compounds"][oxide_features]
missing = compounds.isna().sum()
print(missing)

rows_with_missing = compounds.isna().any(axis=1).mean()
print(f"\nRows with at least one missing oxide: {rows_with_missing:.1%}")

In [ ]:
# For each row, what % of the composition is covered by our 18 oxides?
coverage = df_full["compounds"][oxide_features].sum(axis=1)
print(coverage.describe())
print(f"\nRows with <90% coverage: {(coverage < 90).mean():.1%}")
print(f"Rows with <50% coverage: {(coverage < 50).mean():.1%}")

In [ ]:
# Filter to oxide-dominant compositions
coverage = df_full["compounds"][oxide_features].sum(axis=1)

oxide_mask = coverage > 90

print(f"Rows kept: {oxide_mask.sum()} ({oxide_mask.mean():.1%} of total)")

# Build clean dataframe
clean_df = df_full["compounds"][oxide_features][oxide_mask].copy()
clean_df["other_oxides"] = 100 - coverage[oxide_mask]
clean_df["Tg"] = df_full["property"]["Tg"][oxide_mask]

# Check how many have Tg
print(f"Rows with Tg: {clean_df['Tg'].notna().sum()}")
print(clean_df.head())

In [ ]:
import numpy as np

# Clip negative floating point errors
clean_df["other_oxides"] = clean_df["other_oxides"].clip(lower=0)

# Drop rows without Tg
tg_df = clean_df.dropna(subset=["Tg"]).copy()

print(f"Final dataset shape: {tg_df.shape}")
print(f"\nTg stats (Kelvin):")
print(tg_df["Tg"].describe())
print(f"\nTg stats (Celsius):")
print((tg_df["Tg"] - 273.15).describe())

In [ ]:
tg_df["Tg"] = tg_df["Tg"] - 273.15
print(tg_df["Tg"].describe())

In [3]:
import sys
sys.path.append("..")
from src.data import load_glass_data

df = load_glass_data(target="Tg")
print(df.shape)
print(df.head())

ModuleNotFoundError: No module named 'src'